see source at [mmaction2/demo](https://github.com/open-mmlab/mmaction2/blob/main/demo/demo.py#L34)

In [7]:
%matplotlib inline

from __future__ import annotations

import os
import argparse
from typing import Optional, Tuple

from mmengine import Config, DictAction

In [9]:
def parse_args(argument:list[str]|None=None):
    """ Parsing input arguments
    Args:
        argument (list[str], optional): List of input arguments, each element is for one argument
    Returns:
        (argparse.Namespace): Argument namespace object
    """
    parser=argparse.ArgumentParser(description='MMAction2 demo')
    parser.add_argument('config', help='test config file')
    parser.add_argument('checkpoint', help='checkpoint file/url')
    parser.add_argument('video', help='video file/url or rawframe directory')
    parser.add_argument('label', help='label file')
    parser.add_argument('--device', type=str, default='cpu', choices=['cuda', 'cpu'], help='CPU/CUDA device option')
    parser.add_argument('--fps', default=30, type=int, help='specify fps value of the output video when using rawframes to generate file')
    parser.add_argument('--font-scale', default=12, type=float, help='font scale of the text in the output video')
    parser.add_argument('--font-color', default='white',help='font color of the text in output video')
    parser.add_argument('--target-resolution', nargs=2, default=None, type=int, help='target resolution (width, height) for resizing the '
                       'frames when using a video as input. If either dimension is set to -1, the frames are resized by keeping the existing '
                       'aspect ratio')
    parser.add_argument('--out-filename', default=None, help='output filename')
    
    return parser.parse_args(argument)
    
output_dirpath='D:/results/ucf101'
config_fpath='../config/slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb.py'
checkpoint_fpath=f'{output_dirpath}/mmaction2-slowfast/demo/slowfast_r50_8xb8-4x16x1-256e_kinetics400.pth' # actually slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb_20220901-701b0f6f.pth
output_fpath=f'{output_dirpath}/mmaction2-slowfast/demo/output_notebook.mp4'

argument=f""" {config_fpath} {checkpoint_fpath} {output_dirpath}/demo.mp4
{output_dirpath}/label_map_k400.txt  --out-filename {output_fpath} --font-scale 12  --device cpu 
"""
args=parse_args(argument=argument.split())
cfg=Config.fromfile(args.config)

In [12]:
for k, v in cfg.items():
    print(k, '\t', v, '\n')

model 	 {'type': 'Recognizer3D', 'backbone': {'type': 'ResNet3dSlowFast', 'pretrained': None, 'resample_rate': 8, 'speed_ratio': 8, 'channel_ratio': 8, 'slow_pathway': {'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': True, 'conv1_kernel': (1, 7, 7), 'dilations': (1, 1, 1, 1), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'inflate': (0, 0, 1, 1), 'norm_eval': False}, 'fast_pathway': {'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': False, 'base_channels': 8, 'conv1_kernel': (5, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'norm_eval': False}}, 'cls_head': {'type': 'SlowFastHead', 'in_channels': 2304, 'num_classes': 400, 'spatial_type': 'avg', 'dropout_ratio': 0.5, 'average_clips': 'prob'}, 'data_preprocessor': {'type': 'ActionDataPreprocessor', 'mean': [123.675, 116.28, 103.53], 'std': [58.395, 57.12, 57.375], 'format_shape': 'NCTHW'}} 

default_scope 	 mmaction 

default_hooks 	 {'runtime_info': {'type': 'RuntimeInfoHook'}, 'timer': {'type': 'IterTimerHoo

In [15]:
print(cfg.model.type)
print(cfg.model)

Recognizer3D
{'type': 'Recognizer3D', 'backbone': {'type': 'ResNet3dSlowFast', 'pretrained': None, 'resample_rate': 8, 'speed_ratio': 8, 'channel_ratio': 8, 'slow_pathway': {'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': True, 'conv1_kernel': (1, 7, 7), 'dilations': (1, 1, 1, 1), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'inflate': (0, 0, 1, 1), 'norm_eval': False}, 'fast_pathway': {'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': False, 'base_channels': 8, 'conv1_kernel': (5, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'norm_eval': False}}, 'cls_head': {'type': 'SlowFastHead', 'in_channels': 2304, 'num_classes': 400, 'spatial_type': 'avg', 'dropout_ratio': 0.5, 'average_clips': 'prob'}, 'data_preprocessor': {'type': 'ActionDataPreprocessor', 'mean': [123.675, 116.28, 103.53], 'std': [58.395, 57.12, 57.375], 'format_shape': 'NCTHW'}}


In [22]:
cfg.model.backbone

{'type': 'ResNet3dSlowFast',
 'pretrained': None,
 'resample_rate': 8,
 'speed_ratio': 8,
 'channel_ratio': 8,
 'slow_pathway': {'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': True,
  'conv1_kernel': (1, 7, 7),
  'dilations': (1, 1, 1, 1),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'inflate': (0, 0, 1, 1),
  'norm_eval': False},
 'fast_pathway': {'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': False,
  'base_channels': 8,
  'conv1_kernel': (5, 7, 7),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'norm_eval': False}}

In [21]:
def is_from(module, pkg_name):
    # check whether the backbone is from pkg
    model_type=module['type']
    if isinstance(model_type, str): return model_type.startswith(pkg_name)
    elif inspect.isclass(model_type) or inspect.isfunction(model_type):
        module_name=model_type.__module__
        return pkg_name in module_name
    else: raise TypeError(f"Unsupported types of module {type(module['type'])}")

is_from(cfg.model.backbone, 'mmaction')

False